# AlphaFold2 Structure Prediction

![AlphaFold2 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/alphafold2/hero.png)

This notebook demonstrates protein structure prediction using AlphaFold2 via the ColabDesign JAX wrapper. We cover three common workflows with `run_alphafold2`: single-sequence prediction without MSA for fast screening, multi-chain complex prediction using the Multimer extension, and MSA-assisted prediction that leverages evolutionary information from ColabFold search for higher accuracy.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold2")
display_overview("alphafold2")
display_docs_section("alphafold2", "Background")

## Available tools

In [ ]:
display_available_tools("alphafold2")

### `run_alphafold2`

AlphaFold2 predicts protein 3D structures from amino acid sequences using a deep learning architecture that combines multiple sequence alignments, pairwise distance representations, and recycling iterations to generate near-experimental-accuracy coordinates. The model supports single-chain proteins and multi-chain complexes via the Multimer extension. Optional MSA generation via ColabFold search provides evolutionary context from homologous sequences, substantially improving confidence scores. The `num_recycles` parameter controls how many times predictions are iteratively refined, and the `model_num` parameter selects among five independently trained parameter sets.

In [ ]:
from pathlib import Path
from proto_tools import Mmseqs2HomologySearchConfig, AlphaFold2Input, AlphaFold2Config, run_alphafold2

In [ ]:
# Display input docs
display_api_reference("alphafold2", "input", "run_alphafold2")

# Aequorea victoria GFP (UniProt P42212) — 238-residue β-barrel fluorescent
# protein, classic folding benchmark with dense MSA coverage. Reused below
# in the MSA-assisted example.
protein_sequence = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF"
    "ICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGY"
    "VQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILG"
    "HKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQ"
    "NTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGIT"
    "HGMDELYK"
)
inputs = AlphaFold2Input(complexes=[protein_sequence])

In [ ]:
# Display config docs
display_api_reference("alphafold2", "config", "run_alphafold2")

# Configure AlphaFold2
config = AlphaFold2Config(
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run structure prediction
result = run_alphafold2(inputs, config)

In [ ]:
# Display output docs
display_api_reference("alphafold2", "output", "run_alphafold2")

structure = result.structures[0]

# Print confidence metrics
print(f"  Length: {structure.num_residues} residues")
print(f"  Average pLDDT: {structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {structure.metrics.ptm:.3f}")

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# Visualize predicted structure (auto-coloured by pLDDT)
structure.visualize()

#### Multi-chain complex prediction

When a complex contains more than one chain, AlphaFold2 automatically uses the Multimer model parameters. The iPTM score provides an additional measure of interface quality beyond the global pTM, which is especially useful for evaluating whether the predicted inter-chain contacts are reliable.

In [ ]:
# Insulin heterodimer: the mature A and B chains form a real two-chain complex.
chain_a = "GIVEQCCTSICSLYQLENYCN"  # insulin A chain
chain_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"  # insulin B chain

# Create multi-chain input
inputs_complex = AlphaFold2Input(complexes=[[chain_a, chain_b]])

# Configure (multimer mode is automatic when >1 chain)
config_complex = AlphaFold2Config(
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Run prediction
result_complex = run_alphafold2(inputs_complex, config_complex)
complex_structure = result_complex.structures[0]

# Print metrics (iPTM is available for multimer)
print(f"  Chains: {complex_structure.num_chains}")
print(f"  Average pLDDT: {complex_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {complex_structure.metrics.ptm:.3f}")
print(f"  iPTM score: {complex_structure.metrics.get('iptm', 'N/A')}")

# Visualize the complex with each chain in a distinct colour
complex_structure.visualize(color_by="chain")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column in exported files contains normalized pLDDT confidence scores (0.0 to 1.0), so coloring by B-factor in a viewer directly shows model confidence per residue.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="alphafold2_structure", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'alphafold2_structure.pdb'}")